In [5]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from langchain.tools import tool
from typing import Annotated, TypedDict
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
llm = ChatMistralAI()

In [7]:
loader = PyPDFLoader("file.pdf")
docs = loader.load()

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
chunks = splitter.split_documents(docs)

In [ ]:
emb = MistralAIEmbeddings()
vector_store = FAISS.from_documents(chunks, emb)


In [ ]:
retriever = vector_store.as_retriever({
    "search_type": "similarity", 
    "search_kwargs" : {"k": 5}
})

In [ ]:
@tool
def rag_tool(query: str) -> dict:
    """  """
    result = retriever.invoke(query)
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    return {
        "query": query,
        "context": context,
        "metadata": metadata
    }


In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    message = state["messages"]
    result = llm_with_tools.invoke(message)
    return {"messages": [result]}

tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile()

result = chatbot.invoke({
    "messages": [HumanMessage(content="")]
})
print(result["messages"][-1].content)